In [49]:
import string
import random
import os
os.makedirs("test_output", exist_ok = True)
alphabet = string.ascii_lowercase

In [50]:
def msg_prep(message):
    prepped = ""
    for i in range(len(message)):
        if message[i] in string.ascii_lowercase or message[i] in string.ascii_uppercase:
            prepped += message[i]
    return prepped

# Function to shift a character by a given amount
def shift_char(ch,shift):
    if ch.islower():
        base = ord("a")     # ord("a") finds the ASCII number for "a" 
    else:
        base = ord("A")     # If the letter is uppercase we take the ASCII for "A"
    return chr((ord(ch) - base + shift) % 26 + base)  # Returns the shifted letter. % 26 because we need the alphabet to wrap around

# Caesar cypher encryption and decryption
def caesar_encrypt(message, shift):
    prepped_msg = msg_prep(message)
    if not 0 <= shift <= 25:        # A shift of 26 is just a shift of 0
        raise ValueError("Error, shift must be between 0 and 25.")
    return "".join(shift_char(ch, shift) for ch in prepped_msg)     # Shifts each character in message and joins them with no spaces

def caesar_decrypt(ciphertext, shift):
    if not ciphertext.isalpha():
        raise ValueError("The ciphertext must contain only letters")
    if not 0 <= shift <= 25:
        raise ValueError("Error, shift must be between 0 and 25.")
    return "".join(shift_char(ch, -shift) for ch in ciphertext) # -shift since we are going back to what it was, so subtract the shift.

print(caesar_encrypt("hellohowareyou", 20))
print(caesar_decrypt(caesar_encrypt("hellohowareyou", 20), 20))

byffibiqulysio
hellohowareyou


In [51]:
# Function to convert the characters in the key to individual shifts
def key_to_shifts(key):
    keylist = []
    for ch in key:
        keylist.append(ord(ch.lower()) - ord("a"))  # ch.lower makes ch lowercase so that "KEY" and "key" have the same output
    return keylist

def vigenere_encrypt(message, key):
    prepped_msg = msg_prep(message)
    if not key or not key.isalpha():    # The key needs to be non-empty and be only alphabetical characters
        raise ValueError("There was an error with the key.")
    shifts = key_to_shifts(key)
    return "".join(shift_char(prepped_msg[i], shifts[i % len(key)]) for i in range(len(prepped_msg)))     # Shifts each character from message with the correct shift acquired from the key and joins them

def vigenere_decrypt(ciphertext, key):
    if not ciphertext.isalpha():
        raise ValueError("The ciphertext must contain only letters")
    if not key or not key.isalpha():
        raise ValueError("There was an error with the key.")
    shifts = key_to_shifts(key)
    return "".join(shift_char(ciphertext[i], -shifts[i % len(key)]) for i in range(len(ciphertext)))      # -shifts because we need to shift backwards to the original text

print(vigenere_encrypt("hello", "key"))
print(vigenere_decrypt(vigenere_encrypt("hello", "key"), "key"))

rijvs
hello


In [52]:
# Brief asks that encryption and decryption be implemented to and from files

# Strips everything but letters in a text
def extract_text(text):
    return "".join(ch for ch in text if ch in string.ascii_letters)

# Opens a file and returns the text in it as a string
def read_file(path):
    with open(path, "r", encoding = "utf-8") as f:
        return f.read()

# Saves a string to a file
def write_file(path, text):
    with open(path, "w", encoding = "utf-8") as f:
        f.write(text)

# Define functions which perform the two forms of encryption and decryption on text files
def caesar_encrypt_file(input_file, output_file, shift):
    file_to_string =  read_file(input_file)
    write_file(output_file, caesar_encrypt(extract_text(file_to_string), shift))

def caesar_decrypt_file(input_file, output_file, shift):
    file_to_string =  read_file(input_file)
    write_file(output_file, caesar_decrypt(file_to_string, shift))

def vigenere_encrypt_file(input_file, output_file, key):
    file_to_string =  read_file(input_file)
    write_file(output_file, vigenere_encrypt(extract_text(file_to_string), key))

def vigenere_decrypt_file(input_file, output_file, key):
    file_to_string =  read_file(input_file)
    write_file(output_file, vigenere_decrypt(file_to_string, key))

In [53]:
# Random shifts for Caesar cypher testing
def random_caesar_shift():
    return random.randint(0,25)

# Random keys for Vigenere cypher testing
def random_vigenere_keys(length):
    return "".join(random.choice(alphabet) for _ in range(length))

# Returns a random message made up of lower and uppercase letters of a specified length
def random_message(length):
    return "".join(random.choice(string.ascii_letters) for _ in range(length))

def test_random(trials = 100):
    for _ in range(trials):
        r_msg = random_message(random.randint(1, 200))
        r_shift = random_caesar_shift()
        r_key = random_vigenere_keys(random.randint(1, len(r_msg)))
        assert caesar_decrypt(caesar_encrypt(r_msg, r_shift), r_shift) == r_msg
        assert vigenere_decrypt(vigenere_encrypt(r_msg, r_key), r_key) == r_msg
        assert vigenere_encrypt(r_msg, alphabet[r_shift]) == caesar_encrypt(r_msg, r_shift)   # A single-letter key should behave like the Caesar cypher
    print(f"Random tests: passed {trials} trials.")


def test_caesar(message):
    for shift in range(26):
        assert caesar_decrypt(caesar_encrypt(message, shift), shift) == message
    print(f"Caesar: Passed all 26 shifts.")

test_caesar("HelloWorld")
test_random()

def test_files(file):
    original = extract_text(read_file(file))
    r_shift = random_caesar_shift()
    caesar_encrypt_file(file, "test_output/caesar_encrypted.txt", r_shift)
    caesar_decrypt_file("test_output/caesar_encrypted.txt", "test_output/caesar_decrypted.txt", r_shift)
    assert read_file("test_output/caesar_decrypted.txt") == original

    r_key = random_vigenere_keys(random.randint(1, len(original)))
    vigenere_encrypt_file(file, "test_output/vigenere_encrypted.txt", r_key)
    vigenere_decrypt_file("test_output/vigenere_encrypted.txt", "test_output/vigenere_decrypted.txt", r_key)
    assert read_file("test_output/vigenere_decrypted.txt") == original
    print(f"File tests passed for {file}")

test_files("message_moby_dick.txt")
test_files("message_wuthering_heights.txt")

Caesar: Passed all 26 shifts.
Random tests: passed 100 trials.
File tests passed for message_moby_dick.txt
File tests passed for message_wuthering_heights.txt


In [54]:
frequencies = {'count': 0}

def freq_calc(message):
    occurrences = {}
    freq_single = {}
    for i in range(len(alphabet)):
        occurrences[alphabet[i]] = 0
    message = msg_prep(message).lower()
    for i in range(len(message)):
        occurrences[message[i]] += 1
    for vals in occurrences:
        freq_single[vals] = occurrences[vals] / len(message)
    return freq_single

def freq_archive(message):
    global frequencies
    frequencies['count'] += 1
    freq_single = freq_calc(message)
    for vals in freq_single:
        if vals not in frequencies:
            frequencies[vals] = freq_single[vals]
        else:
            frequencies[vals] = frequencies[vals] + (freq_single[vals] - frequencies[vals]) / frequencies['count']
    return frequencies

In [55]:
freq_archive(extract_text(read_file("message_emma.txt")))
freq_archive(extract_text(read_file("message_heart_of_darkness.txt")))
freq_archive(extract_text(read_file("message_moby_dick.txt")))
freq_archive(extract_text(read_file("message_tomorrows_tangle.txt")))
freq_archive(extract_text(read_file("message_treasure_island.txt")))
freq_archive(extract_text(read_file("message_wuthering_heights.txt")))

{'count': 6,
 'a': 0.01703102825364949,
 'b': 0.0029920597347558252,
 'c': 0.005454137784308012,
 'd': 0.00980838351486881,
 'e': 0.026928565586011487,
 'f': 0.004975145526808943,
 'g': 0.004720602039528271,
 'h': 0.012718543389020405,
 'i': 0.015110640531286898,
 'j': 0.00030152204095188487,
 'k': 0.0015873510864610333,
 'l': 0.008890688185511858,
 'm': 0.005671526178459993,
 'n': 0.015493863983072195,
 'o': 0.01569953282346377,
 'p': 0.004123369011339089,
 'q': 0.00022130314386731526,
 'r': 0.012048314413026945,
 's': 0.013729811778097933,
 't': 0.018710595402763836,
 'u': 0.006410563778315333,
 'v': 0.0024303594916289723,
 'w': 0.0043456206114685335,
 'x': 0.00036220141944488414,
 'y': 0.0040678029502497865,
 'z': 5.5356230527385806e-05}

In [56]:
def freq_rank(message):
    ranked_freqs = {}
    freq_single = freq_calc(message)
    freq_array = sorted(freq_single.items(), key=lambda x:x[1])
    letter_list = [f[0] for f in freq_array]
    for i in range(len(freq_array)):
        ranked_freqs[i] = letter_list[i]
    return ranked_freqs

def freq_rank_archive(frequencies):
    ranked_archive = {}
    freq_array = sorted(frequencies.items(), key=lambda x:x[1])
    letter_list = [f[0] for f in freq_array]
    for i in range(len(freq_array)):
        ranked_archive[i] = letter_list[i]
    return ranked_archive

def shift_break(message):
    ranked_archive = freq_rank_archive(frequencies)
    ranked_freqs = freq_rank(message)
    for i in range(len(alphabet)):
        shift = (ord(ranked_freqs[i]) - ord(ranked_archive[i]))%26
    return shift

In [57]:
#taken and modified from task 1
def task2_test_files(file):
    original = extract_text(read_file(file))
    r_shift = random_caesar_shift()
    caesar_encrypt_file(file, "test_output/task2_caesar_encrypted.txt", r_shift)
    caesar_decrypt_file("test_output/task2_caesar_encrypted.txt", "test_output/task2_caesar_decrypted.txt", shift_break(original))
    assert read_file("test_output/task2_caesar_decrypted.txt") == original
    print(f"File tests passed for {file}")

task2_test_files("message_moby_dick.txt")
task2_test_files("message_wuthering_heights.txt")

AssertionError: 